In [1]:
!pip install pytorch-lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.9.90
    Uninstalling nvidia-curand-cu12-1

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
import nibabel as nib
import random
from collections import defaultdict
import time
import matplotlib.pyplot as plt
from torch.nn.utils import spectral_norm
import glob

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị đang sử dụng: {device}")

# Giải phóng VRAM
torch.cuda.empty_cache()

# Kiểm tra dung lượng ổ đĩa
def check_disk_space():
    stat = os.statvfs('/kaggle/working/')
    free_space = stat.f_bavail * stat.f_frsize / 1024**3
    print(f"Dung lượng trống: {free_space:.2f}GB")
    return free_space

check_disk_space()
if check_disk_space() < 1:
    print("Cảnh báo: Dung lượng ổ đĩa thấp, xóa file cũ...")
    for f in glob.glob("/kaggle/working/*.pth"):
        os.remove(f)
    for f in glob.glob("/kaggle/working/*.png"):
        os.remove(f)

# Xử lý participants.xlsx: Xóa hai subject không mong muốn
participants_file_input = "/kaggle/input/data-gan/participants.xlsx"
participants_file_output = "/kaggle/working/processed_mri_slices/participants.xlsx"

# Đọc file gốc
participants_df = pd.read_excel(participants_file_input)

# Xóa hai subject không mong muốn
subjects_to_remove = ['sub-BrainAge019983', 'sub-BrainAge005600']
participants_df = participants_df[~participants_df['subject_id'].isin(subjects_to_remove)]

# Tạo thư mục đích nếu chưa tồn tại
os.makedirs(os.path.dirname(participants_file_output), exist_ok=True)

# Lưu file đã chỉnh sửa
participants_df.to_excel(participants_file_output, index=False)
print(f"Đã xóa {len(subjects_to_remove)} subject và lưu file participants.xlsx tại {participants_file_output}")

# Đọc thông tin subject từ participants.xlsx đã chỉnh sửa
subject_info = {}
for _, row in participants_df.iterrows():
    subject_id = row['subject_id']
    subject_info[subject_id] = {
        'age': float(row['subject_age']),
        'gender': 1 if row['subject_sex'] == 'm' else 0
    }

# Tạo chỉ mục cho truy vấn real_B, giới hạn tuổi 20-80
data_dir = "/kaggle/input/data-gan"  # Đường dẫn mới
subjects = [s for s in os.listdir(data_dir) if s.startswith('sub-BrainAge')]
ages = [subject_info[s]['age'] for s in subjects]
genders = [subject_info[s]['gender'] for s in subjects]
min_age = 20
max_age = 80
ages_normalized = (np.array(ages) - min_age) / (max_age - min_age)
condition_cache = defaultdict(list)
for idx, subject_id in enumerate(subjects):
    age = int(ages[idx])
    if min_age <= age <= max_age:
        gender = genders[idx]
        condition_cache[(age, gender)].append(idx)

# Kiểm tra ảnh gốc với xử lý kích thước
def check_original_mri(data_dir, subjects):
    subject_id = subjects[0]
    subject_dir = os.path.join(data_dir, subject_id, "anat")
    file_path = os.path.join(subject_dir, f"{subject_id}_T1w_axial.nii")
    mri_img = nib.load(file_path)
    mri_data = mri_img.get_fdata()
    # Lấy mảng 2D bằng cách loại bỏ chiều thừa (nếu có)
    if mri_data.ndim == 3 and mri_data.shape[0] == 1:
        mri_data = mri_data[0]
    elif mri_data.ndim == 3 and mri_data.shape[0] > 1:
        mri_data = mri_data[0]  # Lấy slice đầu tiên nếu là chuỗi
    plt.figure()
    plt.imshow(mri_data, cmap='gray')
    plt.title(f"Original MRI Axial ({subject_id})")
    plt.axis('off')
    plt.savefig('/kaggle/working/original_slice.png')
    plt.close()
    print("Đã lưu ảnh gốc tại /kaggle/working/original_slice.png")

# Dataset với preload và xử lý kích thước
class MRIDataset(Dataset):
    def __init__(self, data_dir, subjects, subject_info, target_shape=(128, 128)):
        self.data_dir = data_dir
        self.subjects = subjects
        self.subject_info = subject_info
        self.target_shape = target_shape
        self.data = []
        
        print("Preloading dataset...")
        for subject_id in subjects:
            subject_dir = os.path.join(data_dir, subject_id, "anat")
            slices = []
            for slice_type in ['axial', 'coronal', 'sagittal']:
                file_path = os.path.join(subject_dir, f"{subject_id}_T1w_{slice_type}.nii")
                mri_img = nib.load(file_path)
                mri_data = mri_img.get_fdata()
                # Xử lý kích thước
                if mri_data.ndim == 3 and mri_data.shape[0] == 1:
                    mri_data = mri_data[0]
                elif mri_data.ndim == 3 and mri_data.shape[0] > 1:
                    mri_data = mri_data[0]  # Lấy slice đầu tiên
                if np.any(np.isnan(mri_data)) or np.any(np.isinf(mri_data)):
                    print(f"Cảnh báo: MRI {file_path} chứa nan/inf")
                    mri_data = np.nan_to_num(mri_data, nan=0.0, posinf=1.0, neginf=-1.0)
                mri_normalized = (mri_data - mri_data.min()) / (mri_data.max() - mri_data.min() + 1e-8)
                mri_normalized = mri_normalized * 2 - 1  # [-1, 1]
                slices.append(mri_normalized)
            mri_tensor = torch.tensor(np.stack(slices), dtype=torch.float32)
            if torch.any(torch.isnan(mri_tensor)) or torch.any(torch.isinf(mri_tensor)):
                print(f"Cảnh báo: MRI tensor chứa nan/inf cho subject {subject_id}")
            age = (subject_info[subject_id]['age'] - min_age) / (max_age - min_age) if min_age <= subject_info[subject_id]['age'] <= max_age else 0
            gender = subject_info[subject_id]['gender']
            condition = torch.tensor([age, gender], dtype=torch.float32)
            self.data.append((mri_tensor, condition))
        print("Dataset preloaded.")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# Hàm truy vấn ảnh
def get_image_and_condition(idx, dataset):
    mri_tensor, condition = dataset[idx]
    return mri_tensor.to(device), condition.to(device)

# Tạo Dataset và DataLoader
dataset = MRIDataset(data_dir, subjects, subject_info, target_shape=(128, 128))
check_original_mri(data_dir, subjects)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

Thiết bị đang sử dụng: cuda
Dung lượng trống: 19.50GB
Dung lượng trống: 19.50GB
Đã xóa 2 subject và lưu file participants.xlsx tại /kaggle/working/processed_mri_slices/participants.xlsx
Preloading dataset...
Dataset preloaded.
Đã lưu ảnh gốc tại /kaggle/working/original_slice.png


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
import matplotlib.pyplot as plt
from collections import defaultdict
import random
import nibabel as nib
import pandas as pd
from PIL import Image
import time

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị đang sử dụng: {device}")

# Định nghĩa hằng số
LATENT_DIM = 512
NUM_MAPPING_LAYERS = 8
NUM_STYLE_LAYERS = 14
IMAGE_SIZE = 128
NUM_CHANNELS = 3
CONDITION_DIM = 2

# Xác định class weights và sizes cho từng resolution
def get_layer_dimensions(image_size=128):
    resolutions = []
    channels = []
    res = 4
    channel = 512
    
    while res <= image_size:
        resolutions.append(res)
        channels.append(channel)
        res *= 2
        channel = max(int(channel / 2), 16)
        
    return resolutions, channels

# Các layer cơ bản cho StyleGAN
class WSConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, gain=2):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.scale = (gain / (in_channels * kernel_size ** 2)) ** 0.5
        nn.init.normal_(self.conv.weight)
        nn.init.zeros_(self.conv.bias)
        
    def forward(self, x):
        return self.conv(x * self.scale)

class Noise(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(channels))
        
    def forward(self, x):
        noise = torch.randn((x.shape[0], 1, x.shape[2], x.shape[3]), device=x.device)
        return x + self.weight.view(1, -1, 1, 1) * noise

class AdaIN(nn.Module):
    def __init__(self, channels, style_dim):
        super().__init__()
        self.instance_norm = nn.InstanceNorm2d(channels)
        self.style_scale = nn.Linear(style_dim, channels)
        self.style_bias = nn.Linear(style_dim, channels)
        
    def forward(self, x, style):
        normalized = self.instance_norm(x)
        scale = self.style_scale(style).unsqueeze(2).unsqueeze(3)
        bias = self.style_bias(style).unsqueeze(2).unsqueeze(3)
        return normalized * scale + bias

class MappingNetwork(nn.Module):
    def __init__(self, latent_dim, style_dim, condition_dim, num_layers=8):
        super().__init__()
        layers = []
        in_dim = latent_dim + condition_dim
        for i in range(num_layers):
            layers.append(nn.Linear(in_dim if i == 0 else style_dim, style_dim))
            layers.append(nn.LeakyReLU(0.2))
        self.mapping = nn.Sequential(*layers)
        
    def forward(self, latent, condition):
        x = torch.cat([latent, condition], dim=1)
        return self.mapping(x)

class StyleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, style_dim):
        super().__init__()
        self.conv = WSConv2d(in_channels, out_channels)
        self.noise = Noise(out_channels)
        self.adain = AdaIN(out_channels, style_dim)
        self.activation = nn.LeakyReLU(0.2)
        
    def forward(self, x, style):
        x = self.conv(x)
        x = self.noise(x)
        x = self.adain(x, style)
        return self.activation(x)

class ToRGB(nn.Module):
    def __init__(self, in_channels, style_dim):
        super().__init__()
        self.conv = WSConv2d(in_channels, NUM_CHANNELS, kernel_size=1, padding=0)
        self.adain = AdaIN(NUM_CHANNELS, style_dim)
        
    def forward(self, x, style):
        return self.adain(self.conv(x), style)

class Generator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, style_dim=512, condition_dim=CONDITION_DIM):
        super().__init__()
        self.mapping_network = MappingNetwork(latent_dim, style_dim, condition_dim, NUM_MAPPING_LAYERS)
        self.resolutions, self.channels = get_layer_dimensions()
        self.constant = nn.Parameter(torch.ones(1, self.channels[0], self.resolutions[0], self.resolutions[0]))
        self.style_blocks = nn.ModuleList()
        self.to_rgb = nn.ModuleList()
        self.style_blocks.append(StyleBlock(self.channels[0], self.channels[0], style_dim))
        self.style_blocks.append(StyleBlock(self.channels[0], self.channels[0], style_dim))
        self.to_rgb.append(ToRGB(self.channels[0], style_dim))
        for i in range(1, len(self.resolutions)):
            in_channel = self.channels[i-1]
            out_channel = self.channels[i]
            self.style_blocks.append(StyleBlock(in_channel, out_channel, style_dim))
            self.style_blocks.append(StyleBlock(out_channel, out_channel, style_dim))
            self.to_rgb.append(ToRGB(out_channel, style_dim))
    
    def forward(self, latent, condition, return_latents=False):
        batch_size = latent.shape[0]
        style = self.mapping_network(latent, condition)
        style_mixing_prob = 0.9 if self.training else 0.0
        x = self.constant.repeat(batch_size, 1, 1, 1)
        if self.training and random.random() < style_mixing_prob:
            latent2 = torch.randn_like(latent)
            style2 = self.mapping_network(latent2, condition)
            crossover_point = random.randint(1, len(self.style_blocks))
            styles = [style] * crossover_point + [style2] * (len(self.style_blocks) - crossover_point)
        else:
            styles = [style] * len(self.style_blocks)
        rgb = None
        for i in range(len(self.resolutions)):
            block_idx = i * 2
            x = self.style_blocks[block_idx](x, styles[block_idx])
            x = self.style_blocks[block_idx + 1](x, styles[block_idx + 1])
            curr_rgb = self.to_rgb[i](x, styles[block_idx])
            if rgb is not None:
                rgb = F.interpolate(rgb, scale_factor=2, mode='bilinear', align_corners=False)
                rgb = rgb + curr_rgb
            else:
                rgb = curr_rgb
            if i < len(self.resolutions) - 1:
                x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if return_latents:
            return rgb, styles
        return rgb

class Discriminator(nn.Module):
    def __init__(self, condition_dim=CONDITION_DIM):
        super().__init__()
        self.resolutions, self.channels = get_layer_dimensions()
        self.resolutions = self.resolutions[::-1]
        self.channels = self.channels[::-1]
        self.from_rgb = nn.ModuleList()
        for channel in self.channels:
            self.from_rgb.append(nn.Sequential(
                WSConv2d(NUM_CHANNELS, channel, kernel_size=1, padding=0),
                nn.LeakyReLU(0.2)
            ))
        self.blocks = nn.ModuleList()
        for i in range(len(self.channels) - 1):
            self.blocks.append(nn.Sequential(
                WSConv2d(self.channels[i], self.channels[i]),
                nn.LeakyReLU(0.2),
                WSConv2d(self.channels[i], self.channels[i+1]),
                nn.LeakyReLU(0.2),
                nn.AvgPool2d(2)
            ))
        self.final_block = nn.Sequential(
            WSConv2d(self.channels[-1], self.channels[-1]),
            nn.LeakyReLU(0.2),
            nn.Conv2d(self.channels[-1], self.channels[-1], kernel_size=4, padding=0),
            nn.LeakyReLU(0.2)
        )
        self.output = nn.Linear(self.channels[-1] + condition_dim, 1)
        
    def forward(self, img, condition):
        x = None
        for i in range(len(self.resolutions)):
            if i == 0:
                x = self.from_rgb[i](img)
            else:
                y = F.avg_pool2d(img, 2**i)
                y = self.from_rgb[i](y)
                x = x + y
            if i < len(self.blocks):
                x = self.blocks[i](x)
        x = self.final_block(x)
        x = x.view(x.shape[0], -1)
        x = torch.cat([x, condition], dim=1)
        return self.output(x)

class MRIStyleGAN(pl.LightningModule):
    def __init__(self, 
                 latent_dim=LATENT_DIM, 
                 style_dim=512, 
                 condition_dim=CONDITION_DIM,
                 lr=0.0002, 
                 beta1=0.0, 
                 beta2=0.99,
                 start_lambda_gp=10.0,
                 decay_lambda_gp=False):
        super().__init__()
        self.save_hyperparameters()
        self.automatic_optimization = False
        self.generator = Generator(latent_dim, style_dim, condition_dim)
        self.discriminator = Discriminator(condition_dim)
        self.lambda_gp = start_lambda_gp
        self.decay_lambda_gp = decay_lambda_gp
        self.validation_z = torch.randn(8, latent_dim)
        self.validation_conditions = torch.tensor([
            [0.0, 0], [0.0, 1], [0.2, 0], [0.2, 1],
            [0.5, 0], [0.5, 1], [0.8, 0], [0.8, 1]
        ], dtype=torch.float32)
        self.d_loss_history = []
        self.g_loss_history = []
    
    def configure_optimizers(self):
        g_opt = torch.optim.Adam(
            self.generator.parameters(), 
            lr=self.hparams.lr,
            betas=(self.hparams.beta1, self.hparams.beta2)
        )
        d_opt = torch.optim.Adam(
            self.discriminator.parameters(), 
            lr=self.hparams.lr,
            betas=(self.hparams.beta1, self.hparams.beta2)
        )
        return [d_opt, g_opt]
    
    def gradient_penalty(self, real_imgs, fake_imgs, conditions):
        batch_size = real_imgs.size(0)
        alpha = torch.rand(batch_size, 1, 1, 1, device=self.device)
        interpolated = alpha * real_imgs + (1 - alpha) * fake_imgs
        interpolated.requires_grad_(True)
        d_interpolated = self.discriminator(interpolated, conditions)
        gradients = torch.autograd.grad(
            outputs=d_interpolated,
            inputs=interpolated,
            grad_outputs=torch.ones_like(d_interpolated),
            create_graph=True,
            retain_graph=True,
        )[0]
        gradients = gradients.view(batch_size, -1)
        gradient_norm = gradients.norm(2, dim=1)
        gradient_penalty = ((gradient_norm - 1) ** 2).mean()
        return gradient_penalty
    
    def training_step(self, batch, batch_idx):
        d_opt, g_opt = self.optimizers()
        real_imgs, conditions = batch
        batch_size = real_imgs.shape[0]
        z = torch.randn(batch_size, self.hparams.latent_dim, device=self.device)
        
        # Train Discriminator (3 lần)
        for _ in range(3):
            d_opt.zero_grad()
            with torch.no_grad():
                fake_imgs = self.generator(z, conditions)
            real_pred = self.discriminator(real_imgs, conditions)
            fake_pred = self.discriminator(fake_imgs, conditions)
            d_loss = fake_pred.mean() - real_pred.mean()
            gp = self.gradient_penalty(real_imgs, fake_imgs, conditions)
            d_loss = d_loss + self.lambda_gp * gp
            self.manual_backward(d_loss)
            d_opt.step()
        
        # Train Generator
        g_opt.zero_grad()
        fake_imgs = self.generator(z, conditions)
        fake_pred = self.discriminator(fake_imgs, conditions)
        g_loss = -fake_pred.mean()
        self.manual_backward(g_loss)
        g_opt.step()
        
        self.log("d_loss", d_loss, prog_bar=True)
        self.log("g_loss", g_loss, prog_bar=True)
        self.log("gp", gp, prog_bar=False)
        self.d_loss_history.append(d_loss.item())
        self.g_loss_history.append(g_loss.item())
    
    def on_epoch_end(self):
        if self.decay_lambda_gp:
            self.lambda_gp = max(0.1, self.lambda_gp * 0.99)
            self.log("lambda_gp", self.lambda_gp)
    
    def validation_step(self, batch, batch_idx):
        pass
    
    def on_validation_epoch_end(self):
        # Chỉ hiển thị ảnh mỗi 10 epoch
        if (self.current_epoch + 1) % 10 == 0:
            self.generator.eval()
            with torch.no_grad():
                z = self.validation_z.to(self.device)
                conditions = self.validation_conditions.to(self.device)
                samples = self.generator(z, conditions)
                samples = (samples + 1) / 2.0  # [-1, 1] -> [0, 1]
                
                # Định nghĩa tên kênh
                channel_names = ['Axial', 'Coronal', 'Sagittal']
                
                # Tạo hình ảnh cho từng kênh riêng biệt
                min_age, max_age = 20, 80
                for channel_idx in range(NUM_CHANNELS):
                    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
                    axes = axes.flatten()
                    for i, (sample, condition) in enumerate(zip(samples, conditions)):
                        age_normalized = condition[0].item()
                        age = age_normalized * (max_age - min_age) + min_age
                        gender = "Nam" if condition[1].item() == 0 else "Nữ"
                        # Lấy kênh hiện tại
                        channel_data = sample[channel_idx].cpu().numpy()
                        # Hiển thị ảnh grayscale
                        axes[i].imshow(channel_data, cmap='gray')
                        axes[i].set_title(f"Tuổi: {age:.1f}, Giới tính: {gender}")
                        axes[i].axis('off')
                    plt.tight_layout()
                    plt.suptitle(f"{channel_names[channel_idx]} Channel - Epoch {self.current_epoch}", y=1.02)
                    # Lưu ảnh
                    save_path = f"/kaggle/working/{channel_names[channel_idx].lower()}_samples_epoch_{self.current_epoch}.png"
                    plt.savefig(save_path)
                    # Log vào TensorBoard
                    channel_grid = torchvision.utils.make_grid(samples[:, channel_idx:channel_idx+1], nrow=4, normalize=True, padding=10)
                    self.logger.experiment.add_image(f"{channel_names[channel_idx]}_samples", channel_grid, self.current_epoch)
                    plt.show()
                    plt.close()
                
            self.generator.train()
        
        # Plot losses
        if len(self.d_loss_history) > 0 and len(self.g_loss_history) > 0:
            plt.figure(figsize=(10, 5))
            plt.plot(self.d_loss_history, label='D Loss')
            plt.plot(self.g_loss_history, label='G Loss')
            plt.legend()
            plt.savefig(f"/kaggle/working/losses_epoch_{self.current_epoch}.png")
            plt.close()

class MRIDataset(Dataset):
    def __init__(self, data_dir, subjects, subject_info, target_shape=(128, 128)):
        self.data_dir = data_dir
        self.subjects = subjects
        self.subject_info = subject_info
        self.target_shape = target_shape
        self.data = []
        print("Preloading dataset...")
        for subject_id in subjects:
            subject_dir = os.path.join(data_dir, subject_id, "anat")
            slices = []
            for slice_type in ['axial', 'coronal', 'sagittal']:
                file_path = os.path.join(subject_dir, f"{subject_id}_T1w_{slice_type}.nii")
                mri_img = nib.load(file_path)
                mri_data = mri_img.get_fdata()
                if mri_data.ndim == 3 and mri_data.shape[0] == 1:
                    mri_data = mri_data[0]
                elif mri_data.ndim == 3 and mri_data.shape[0] > 1:
                    mri_data = mri_data[0]
                if np.any(np.isnan(mri_data)) or np.any(np.isinf(mri_data)):
                    mri_data = np.nan_to_num(mri_data, nan=0.0, posinf=1.0, neginf=-1.0)
                mri_normalized = (mri_data - mri_data.min()) / (mri_data.max() - mri_data.min() + 1e-8)
                mri_normalized = mri_normalized * 2 - 1
                slices.append(mri_normalized)
            mri_tensor = torch.tensor(np.stack(slices), dtype=torch.float32)
            min_age = 20
            max_age = 80
            age = (self.subject_info[subject_id]['age'] - min_age) / (max_age - min_age)
            age = max(0.0, min(1.0, age))
            gender = self.subject_info[subject_id]['gender']
            condition = torch.tensor([age, gender], dtype=torch.float32)
            self.data.append((mri_tensor, condition))
        print(f"Dataset preloaded with {len(self.data)} samples.")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

def train_stylegan(data_dir, participants_df, batch_size=16, num_epochs=100, lr=0.0002, checkpoint_path=None):
    subject_info = {}
    for _, row in participants_df.iterrows():
        subject_id = row['subject_id']
        subject_info[subject_id] = {
            'age': float(row['subject_age']),
            'gender': 1 if row['subject_sex'] == 'm' else 0
        }
    subjects = [s for s in os.listdir(data_dir) if s.startswith('sub-BrainAge')]
    dataset = MRIDataset(data_dir, subjects, subject_info, target_shape=(128, 128))
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"Loading model from checkpoint: {checkpoint_path}")
        model = MRIStyleGAN.load_from_checkpoint(checkpoint_path)
    else:
        model = MRIStyleGAN(
            latent_dim=LATENT_DIM,
            style_dim=512,
            condition_dim=CONDITION_DIM,
            lr=lr,
            beta1=0.0,
            beta2=0.99,
            start_lambda_gp=10.0,
            decay_lambda_gp=True
        )
    checkpoint_callback = ModelCheckpoint(
        monitor="g_loss",
        dirpath="/kaggle/working/checkpoints/",
        filename="stylegan-mri-{epoch:02d}",
        save_top_k=3,
        mode="min",
        every_n_epochs=5,
        save_last=True
    )
    lr_monitor = LearningRateMonitor(logging_interval="epoch")
    logger = TensorBoardLogger("/kaggle/working/", name="stylegan_logs")
    trainer = pl.Trainer(
        max_epochs=num_epochs,
        accelerator="auto",
        devices=1 if torch.cuda.is_available() else None,
        logger=logger,
        callbacks=[checkpoint_callback, lr_monitor],
        log_every_n_steps=10,
        check_val_every_n_epoch=1
    )
    trainer.fit(model, train_loader, val_loader, ckpt_path=checkpoint_path if checkpoint_path else None)
    return model

def generate_samples(model, num_samples=8, specified_ages=None, specified_genders=None):
    model.eval()
    z = torch.randn(num_samples, LATENT_DIM, device=model.device)
    if specified_ages is not None and specified_genders is not None:
        conditions = []
        for age, gender in zip(specified_ages, specified_genders):
            conditions.append([age, gender])
        conditions = torch.tensor(conditions, dtype=torch.float32, device=model.device)
    else:
        conditions = torch.tensor([
            [0.0, 0], [0.0, 1], [0.3, 0], [0.3, 1],
            [0.6, 0], [0.6, 1], [1.0, 0], [1.0, 1]
        ], dtype=torch.float32, device=model.device)
    with torch.no_grad():
        samples = model.generator(z, conditions)
        samples = (samples + 1) / 2.0
    channel_names = ['Axial', 'Coronal', 'Sagittal']
    min_age, max_age = 20, 80
    for channel_idx in range(NUM_CHANNELS):
        fig, axes = plt.subplots(2, 4, figsize=(12, 12))
        axes = axes.flatten()
        for i, (sample, condition) in enumerate(zip(samples, conditions)):
            age_normalized = condition[0].item()
            age = age_normalized * (max_age - min_age) + min_age
            gender = "Nam" if condition[1].item() == 0 else "Nữ"
            channel_data = sample[channel_idx].cpu().numpy()
            axes[i].imshow(channel_data, cmap='gray')
            axes[i].set_title(f"Tuổi: {age:.1f}, Giới tính: {gender}")
            axes[i].axis('off')
        plt.tight_layout()
        plt.suptitle(f"{channel_names[channel_idx]} Channel - Generated Samples", y=1.02)
        plt.savefig(f"/kaggle/working/generated_{channel_names[channel_idx].lower()}_samples.png")
        plt.close()
    return samples

if __name__ == "__main__":
    participants_file = "/kaggle/working/processed_mri_slices/participants.xlsx"
    participants_df = pd.read_excel(participants_file)
    data_dir = "/kaggle/input/data-gan"
    model = train_stylegan(
        data_dir=data_dir,
        participants_df=participants_df,
        batch_size=16,
        num_epochs=100,
        lr=0.0002
    )
    ages = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 0.5, 0.5]
    genders = [0, 0, 0, 0, 0, 0, 0, 1]
    samples = generate_samples(model, specified_ages=ages, specified_genders=genders)
    print("Training và generating samples đã hoàn thành!")

Thiết bị đang sử dụng: cuda
Preloading dataset...
Dataset preloaded with 4948 samples.


2025-05-21 12:51:19.790338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747831879.984696      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747831880.046208      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]